<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_14_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 14: Learning to Write Like Shakespeare
### Long Short-Term Memory (LSTM)

Standard RNNs struggle with long-range dependencies due to vanishing gradients. LSTMs solve this with a gating mechanism that explicitly controls what information to remember, forget, and output.

## 1. The Vanishing Gradient Problem (Review)

In a standard RNN, the gradient is multiplied by the same weights at every time step. After many steps, the gradient becomes too small to update early weights.

In [ ]:
import numpy as np

np.random.seed(0)

W = np.random.rand(5, 5) * 0.5
grad = np.ones(5)

print("Gradient magnitude over backprop steps (standard RNN):")
print(f"{'Step':<8} {'||grad||':>12}")
print("-" * 22)

for t in range(10):
    norm = np.linalg.norm(grad)
    print(f"{t:<8} {norm:>12.6f}")
    grad = W.T.dot(grad)

print("\nGradient shrinks rapidly -> early layers learn nothing.")

## 2. The LSTM Cell: Four Gates

An LSTM cell maintains two states: the **cell state** (long-term memory) and the **hidden state** (short-term memory). Four learned gates control information flow:

- **Forget gate**: what to erase from cell state
- **Input gate**: what new information to write
- **Gate gate** (cell gate): candidate values to write
- **Output gate**: what to expose as the hidden state

In [ ]:
import numpy as np

def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x):    return np.tanh(x)

np.random.seed(0)

input_size  = 4
hidden_size = 3
concat_size = input_size + hidden_size

W_f = np.random.rand(concat_size, hidden_size) * 0.1
W_i = np.random.rand(concat_size, hidden_size) * 0.1
W_g = np.random.rand(concat_size, hidden_size) * 0.1
W_o = np.random.rand(concat_size, hidden_size) * 0.1

def lstm_step(x_t, h_prev, c_prev):
    combined = np.concatenate([x_t, h_prev])
    f = sigmoid(combined.dot(W_f))
    i = sigmoid(combined.dot(W_i))
    g = tanh(combined.dot(W_g))
    o = sigmoid(combined.dot(W_o))
    c = f * c_prev + i * g
    h = o * tanh(c)
    return h, c, {"f": f, "i": i, "g": g, "o": o}

x = np.random.rand(input_size)
h = np.zeros(hidden_size)
c = np.zeros(hidden_size)

h_new, c_new, gates = lstm_step(x, h, c)

print("LSTM gate values (first step):")
for gate_name, gate_val in gates.items():
    print(f"  {gate_name} gate: {np.round(gate_val, 4)}")
print(f"\nNew hidden state: {np.round(h_new, 4)}")
print(f"New cell state  : {np.round(c_new, 4)}")

## 3. Forward Pass Through a Sequence

The LSTM processes a full sequence step by step, updating both `h` and `c` at each step.

In [ ]:
import numpy as np

def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x):    return np.tanh(x)

np.random.seed(1)

vocab     = list("abcde")
char2idx  = {c: i for i, c in enumerate(vocab)}
vocab_size = len(vocab)
hidden_size = 6
concat_size = vocab_size + hidden_size

W_f = np.random.rand(concat_size, hidden_size) * 0.1
W_i = np.random.rand(concat_size, hidden_size) * 0.1
W_g = np.random.rand(concat_size, hidden_size) * 0.1
W_o = np.random.rand(concat_size, hidden_size) * 0.1

def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1
    return v

sequence = "abcde"
h = np.zeros(hidden_size)
c = np.zeros(hidden_size)

print("Processing sequence:", sequence)
print()
for char in sequence:
    x = one_hot(char2idx[char], vocab_size)
    combined = np.concatenate([x, h])
    f = sigmoid(combined.dot(W_f))
    i = sigmoid(combined.dot(W_i))
    g = tanh(combined.dot(W_g))
    o = sigmoid(combined.dot(W_o))
    c = f * c + i * g
    h = o * tanh(c)
    print(f"After '{char}': ||h|| = {np.linalg.norm(h):.4f}")

print(f"\nFinal hidden state shape: {h.shape}")

## 4. Truncated Backpropagation Through Time (TBPTT)

For very long sequences, backpropagating through all time steps is prohibitively expensive. TBPTT splits the sequence into chunks and backpropagates only within each chunk.

In [ ]:
import numpy as np

long_sequence = "abcdeabcdeabcdeabcdeabcde"
chunk_size    = 5

chunks = [long_sequence[i:i+chunk_size] for i in range(0, len(long_sequence), chunk_size)]

print(f"Full sequence length: {len(long_sequence)}")
print(f"Chunk size          : {chunk_size}")
print(f"Number of chunks    : {len(chunks)}")
print()
print("Chunks:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: '{chunk}'  (backprop within this chunk only)")

print("\nHidden state is carried across chunks; gradients are not.")

## 5. Why LSTMs Remember Long-Range Dependencies

The forget gate can be set close to 1, letting the cell state pass through unchanged for many steps. This creates a nearly constant gradient path, unlike the standard RNN where gradients shrink at every step.

In [ ]:
import numpy as np

print("Gradient flow comparison: RNN vs LSTM")
print()
print(f"{'Steps':<8} {'RNN ||grad||':>16} {'LSTM ||grad|| (f=0.99)':>25}")
print("-" * 52)

rnn_grad  = 1.0
lstm_grad = 1.0
W_rnn     = 0.7
f_gate    = 0.99

for t in range(1, 11):
    rnn_grad  *= W_rnn
    lstm_grad *= f_gate
    print(f"{t:<8} {rnn_grad:>16.6f} {lstm_grad:>25.6f}")

print("\nLSTM retains much more gradient across time steps.")